In [1]:
# Standard libraries
import json
import os

# Third-party libraries
import numpy as np
from tqdm import tqdm
import math, glob
import torch

In [2]:
class _PreProcessor:
    """Batch-compatible preprocessor for storm nowcasting features (with `day` removed)."""
    def __init__(self, norm_json: str):
        with open(norm_json, "r") as f:
            self.norm = json.load(f)

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: Tensor of shape (B, 140, 11) or (140, 11)
        returns Tensor of shape (B, 140, 10) or (140, 10)
        """

        # original header: [year, month, day, hour, minute, lat, lon, wp, tir, size, mask]

        # considered header: [month, hour, minute, lat, lon, wp, tir, size, mask]

        is_batch = x.dim() == 3  # (B, 140, 11)

        if not is_batch:
            x = x.unsqueeze(0)  # convert to batch shape

        # Drop 'year' (index 0) and 'day' (index 2 after dropping year)
        x = torch.cat([x[:, :, :1], x[:, :, 2:]], dim=2)  # shape: (B, 140, 9)

        # Restarting index
        month  = x[:, :, 0]
        hour   = x[:, :, 1]
        minute = x[:, :, 2]
        lat    = x[:, :, 3]
        lon    = x[:, :, 4]
        wp     = x[:, :, 5]
        tir    = x[:, :, 6]
        size   = x[:, :, 7]
        mask   = x[:, :, 8]

        B, N = x.shape[:2]
        out = torch.empty((B, N, 10), dtype=torch.float32)

        # month encoding
        out[:, :, 0] = torch.sin(2 * math.pi * (month - 1) / 12.0)
        out[:, :, 1] = torch.cos(2 * math.pi * (month - 1) / 12.0)

        # time of day encoding
        tod = hour + minute / 60.0
        out[:, :, 2] = torch.sin(2 * math.pi * tod / 24.0)
        out[:, :, 3] = torch.cos(2 * math.pi * tod / 24.0)

        # lat, lon, wp, tir and size scaling
        out[:, :, 4] = (lat - self.norm["lat_min"]) / (self.norm["lat_max"] - self.norm["lat_min"])
        out[:, :, 5] = (lon - self.norm["lon_min"]) / (self.norm["lon_max"] - self.norm["lon_min"])
        out[:, :, 6] = torch.log1p(wp) / self.norm["wp_max"]
        out[:, :, 7] = (tir - self.norm["tir_min"]) / (self.norm["tir_max"] - self.norm["tir_min"])
        out[:, :, 8] = torch.log1p(size) / self.norm["size_max"]

        # mask kept as it is
        out[:, :, 9] = mask

        return out if is_batch else out.squeeze(0)


In [3]:
@torch.no_grad()
def build_single_file(
    shards_dir: str,
    norm_path: str,
    out_path: str = "nowcasting_train.pt",
    glob_pattern: str = "*.pt",
):
    """
    Merge multiple shard files (each holding a batch of (x, y) pairs).
    Apply preprocessing once, and write a single output file:
        • inputs  → (N, 140, 10)
        • targets → (N, 1024, 1024)
    """
    processor = _PreProcessor(norm_path)
    shard_files = sorted(glob.glob(os.path.join(shards_dir, glob_pattern)))
    if not shard_files:
        raise FileNotFoundError("No shard files found!")

    xs, ys = [], []
    total = 0
    for fp in tqdm(shard_files, desc="Merging shards"):
        data = torch.load(fp, map_location="cpu")
        x_batch, y_batch = data["inputs"], data["targets"]  # shape: (B, 140, 11), (B, 1024, 1024)

        # preprocess x
        x_batch_proc = processor(x_batch.float())  # shape: (B, 140, 10)
        xs.append(x_batch_proc)
        ys.append(y_batch.float())

    inputs  = torch.stack(xs)  # shape: (N, 140, 10)
    targets = torch.stack(ys)  # shape: (N, 1024, 1024)
    torch.save({"inputs": inputs, "targets": targets}, out_path)

    print(f"✅ Wrote {total:,} samples to {out_path}")

In [4]:
root_dir = "/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Africa_sharded/t1/training"
norm_path = "/home/users/mendrika/EPS-Impact-Case-AI-Nowcasting/model/africa/normalisation/parameters/normalisation.json"
output_path = "/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Africa_sharded/t1/training/preprocessed/nowcasting_train.pt"

In [ ]:
build_single_file(shards_dir=root_dir, 
                  norm_path=norm_path, 
                  out_path=output_path)

Merging shards:   0%|          | 0/1 [00:00<?, ?it/s]